# Using DeterministicModel in Ax: Minimal Working Example

This tutorial demonstrates how to use DeterministicModel for optimization problems where some objectives are analytically known. This addresses common use cases in physical sciences where you want to optimize a tradeoff between hard-to-model performance metrics and analytically computable costs (e.g., monetary cost, computational cost).

## Problem Setup

We'll demonstrate a scenario where:
- **Objective**: Minimize an analytical function (x² + y²) 
- **Constraints**: Multiple black-box constraint functions that are expensive to evaluate

This is a common pattern where the objective function is known analytically (e.g., cost) but the constraints come from complex simulations or experiments.

## Key Components

1. **GenericDeterministicModel**: Wraps analytical functions for use in BoTorch
2. **ModelList**: Combines deterministic and probabilistic models
3. **Modular BoTorch Interface**: Integrates custom models into Ax workflow

In [ ]:
import torch
import numpy as np
from typing import Dict, List, Optional

from ax.core.experiment import Experiment
from ax.core.optimization_config import OptimizationConfig
from ax.core.objective import Objective
from ax.core.parameter import RangeParameter
from ax.core.search_space import SearchSpace
from ax.core.metric import Metric
from ax.core.outcome_constraint import OutcomeConstraint
from ax.core.types import ComparisonOp
from ax.runners.synthetic import SyntheticRunner
from ax.metrics.noisy_function import NoisyFunctionMetric

from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.generator_spec import GeneratorSpec
from ax.generation_strategy.transition_criterion import MinTrials
from ax.adapter.registry import Generators

# BoTorch imports for deterministic models
from botorch.models.deterministic import GenericDeterministicModel
from botorch.models.model import ModelList
from botorch.models import SingleTaskGP

# Modular BoTorch components
from ax.generators.torch.botorch_modular.surrogate import Surrogate
from ax.generators.torch.botorch_modular.model import BoTorchModel
from ax.generators.torch.botorch_modular.acquisition import Acquisition

print("All imports successful!")

## Step 1: Define the Analytical Function and Black-box Constraints

In [ ]:
def analytical_objective(x: torch.Tensor) -> torch.Tensor:
    """
    Analytical objective function: minimize x² + y²
    This could represent cost, energy consumption, or any other 
    analytically computable objective.
    
    Args:
        x: Input tensor of shape (..., 2) where columns are [x, y]
    
    Returns:
        Function values of shape (..., 1)
    """
    return (x**2).sum(dim=-1, keepdim=True)

def constraint_function_1(x: torch.Tensor) -> torch.Tensor:
    """
    Black-box constraint function 1: g1(x,y) = x + y - 0.5 <= 0
    In practice, this could be output from a complex simulation.
    """
    return (x.sum(dim=-1, keepdim=True) - 0.5 + 0.1 * torch.randn(x.shape[:-1] + (1,)))

def constraint_function_2(x: torch.Tensor) -> torch.Tensor:
    """
    Black-box constraint function 2: g2(x,y) = -x + 2*y + 0.2 <= 0
    """
    return (-x[..., 0:1] + 2*x[..., 1:2] + 0.2 + 0.1 * torch.randn(x.shape[:-1] + (1,)))

# Test the functions
test_x = torch.tensor([[0.1, 0.2], [0.3, 0.4]])
print(f"Analytical objective at test points: {analytical_objective(test_x)}")
print(f"Constraint 1 at test points: {constraint_function_1(test_x)}")
print(f"Constraint 2 at test points: {constraint_function_2(test_x)}")

## Step 2: Define Ax Metrics for the Problem

In [ ]:
class AnalyticalMetric(Metric):
    """
    Metric for the analytical objective function.
    This metric will be handled by DeterministicModel.
    """
    def fetch_trial_data(self, trial):
        """Since this is analytical, we don't fetch data - it's computed deterministically."""
        return {}

class ConstraintMetric(NoisyFunctionMetric):
    """
    Metric for constraint functions that will be modeled with GPs.
    """
    def __init__(self, name: str, constraint_func, noise_sd: float = 0.1):
        self.constraint_func = constraint_func
        
        def f(x):
            # Convert from numpy array to tensor for our constraint function
            x_tensor = torch.from_numpy(x).double()
            result = self.constraint_func(x_tensor).numpy()
            return result.flatten()[0] if result.size == 1 else result.flatten()
        
        super().__init__(name=name, f=f, noise_sd=noise_sd)

# Create metrics
analytical_metric = AnalyticalMetric(name="analytical_cost")
constraint_metric_1 = ConstraintMetric(name="constraint_1", constraint_func=constraint_function_1)
constraint_metric_2 = ConstraintMetric(name="constraint_2", constraint_func=constraint_function_2)

print("Metrics defined successfully!")

## Step 3: Set Up the Ax Experiment

In [ ]:
# Define search space
search_space = SearchSpace(
    parameters=[
        RangeParameter(name="x", parameter_type=float, lower=-1.0, upper=1.0),
        RangeParameter(name="y", parameter_type=float, lower=-1.0, upper=1.0),
    ]
)

# Define optimization config
optimization_config = OptimizationConfig(
    objective=Objective(metric=analytical_metric, minimize=True),
    outcome_constraints=[
        OutcomeConstraint(
            metric=constraint_metric_1,
            op=ComparisonOp.LEQ,
            bound=0.0,
            relative=False,
        ),
        OutcomeConstraint(
            metric=constraint_metric_2,
            op=ComparisonOp.LEQ,
            bound=0.0,
            relative=False,
        ),
    ],
)

# Create experiment
experiment = Experiment(
    name="deterministic_model_example",
    search_space=search_space,
    optimization_config=optimization_config,
    runner=SyntheticRunner(),
)

print(f"Experiment created: {experiment.name}")
print(f"Search space: {experiment.search_space}")
print(f"Optimization config: {experiment.optimization_config}")

## Step 4: Create Custom Surrogate with DeterministicModel

This is the key part where we integrate the DeterministicModel into Ax's workflow.

In [ ]:
from ax.generators.torch.botorch_modular.surrogate import Surrogate
from botorch.utils.datasets import SupervisedDataset
from ax.core.search_space import SearchSpaceDigest
from ax.utils.common.typeutils import checked_cast

class MixedDeterministicSurrogate(Surrogate):
    """
    Custom surrogate that combines deterministic and probabilistic models.
    """
    
    def __init__(self, **kwargs):
        # Initialize with empty botorch_model_class since we'll build custom model
        super().__init__(botorch_model_class=ModelList, **kwargs)
        self.analytical_function = analytical_objective
        
    def _construct_model(self, 
                        datasets: List[SupervisedDataset],
                        search_space_digest: SearchSpaceDigest,
                        **kwargs) -> ModelList:
        """
        Construct a ModelList with deterministic model for analytical objective
        and GPs for constraint functions.
        """
        models = []
        
        for i, dataset in enumerate(datasets):
            outcome_name = dataset.outcome_names[0] if dataset.outcome_names else f"outcome_{i}"
            
            if outcome_name == "analytical_cost":
                # Create deterministic model for analytical objective
                det_model = GenericDeterministicModel(f=self.analytical_function)
                models.append(det_model)
                print(f"Created DeterministicModel for {outcome_name}")
            else:
                # Create GP model for constraints
                if len(dataset.X) > 0:  # Only create GP if we have data
                    gp_model = SingleTaskGP(
                        train_X=dataset.X, 
                        train_Y=dataset.Y
                    )
                    models.append(gp_model)
                    print(f"Created SingleTaskGP for {outcome_name}")
                else:
                    # Placeholder - in practice you might handle this differently
                    print(f"Warning: No data for {outcome_name}, skipping")
        
        if not models:
            raise ValueError("No models could be constructed")
        
        return ModelList(*models)
    
    def fit(self, 
            datasets: List[SupervisedDataset],
            search_space_digest: SearchSpaceDigest,
            **kwargs) -> None:
        """
        Fit the surrogate model.
        """
        self._model = self._construct_model(datasets, search_space_digest, **kwargs)
        
        # Fit only the GP models (deterministic models don't need fitting)
        for model in self._model.models:
            if isinstance(model, SingleTaskGP):
                # The GP will be fitted automatically when created
                pass
        
        self._outcomes = []
        for dataset in datasets:
            if dataset.outcome_names:
                self._outcomes.extend(dataset.outcome_names)
        
        print(f"Surrogate fitted with outcomes: {self._outcomes}")

print("Custom surrogate class defined!")

## Step 5: Set Up Generation Strategy with Custom Surrogate

In [ ]:
# Create a custom generator spec that uses our mixed surrogate
custom_generator_spec = GeneratorSpec(
    generator_enum=Generators.BOTORCH_MODULAR,
    model_kwargs={
        "surrogate": MixedDeterministicSurrogate(),
        "botorch_acqf_class": "qExpectedImprovement",  # Simple acquisition function
    },
)

# Create generation strategy
generation_strategy = GenerationStrategy(
    nodes=[
        GenerationNode(
            node_name="sobol_initialization",
            generator_specs=[
                GeneratorSpec(
                    generator_enum=Generators.SOBOL,
                    model_kwargs={"seed": 42},
                ),
            ],
            transition_criteria=[
                MinTrials(
                    threshold=5,  # Start with 5 Sobol points
                    transition_to="bayesian_optimization",
                )
            ],
        ),
        GenerationNode(
            node_name="bayesian_optimization",
            generator_specs=[custom_generator_spec],
        ),
    ]
)

print("Generation strategy created successfully!")

## Step 6: Run the Optimization

Now let's run the optimization and see the DeterministicModel in action!

In [ ]:
from ax.service.scheduler import Scheduler
from ax.service.utils.report_utils import render_report_elements

# Create a simple evaluation function that computes all metrics
def evaluate_trial(parameters: Dict[str, float]) -> Dict[str, float]:
    """
    Evaluation function that computes all metrics for a given parameter configuration.
    """
    x_val = parameters["x"]
    y_val = parameters["y"]
    
    # Convert to tensor for our functions
    x_tensor = torch.tensor([[x_val, y_val]], dtype=torch.double)
    
    # Compute analytical objective (deterministic)
    analytical_value = analytical_objective(x_tensor).item()
    
    # Compute constraints (with noise)
    constraint_1_value = constraint_function_1(x_tensor).item()
    constraint_2_value = constraint_function_2(x_tensor).item()
    
    return {
        "analytical_cost": analytical_value,
        "constraint_1": constraint_1_value,
        "constraint_2": constraint_2_value,
    }

# Let's manually run a few trials to demonstrate
print("Starting optimization...")

# Generate and evaluate initial Sobol trials
for i in range(3):  # Start with fewer trials to demonstrate
    # Generate new trial
    trial = experiment.new_trial(generation_strategy.gen(experiment))
    
    # Evaluate the trial
    parameters = trial.arm.parameters
    results = evaluate_trial(parameters)
    
    # Add data to trial
    raw_data = {
        "metric_names": list(results.keys()),
        "mean": list(results.values()),
        "sem": [0.0] * len(results),  # No standard error for deterministic analytical function
    }
    
    # Complete the trial
    trial.mark_running(no_runner_required=True)
    trial.mark_completed()
    
    print(f"Trial {i+1}: x={parameters['x']:.3f}, y={parameters['y']:.3f}")
    print(f"  Analytical cost: {results['analytical_cost']:.3f}")
    print(f"  Constraint 1: {results['constraint_1']:.3f}")
    print(f"  Constraint 2: {results['constraint_2']:.3f}")
    print()

print(f"Completed {len(experiment.trials)} trials")
print(f"Current generation strategy node: {generation_strategy.current_node_name}")

## Step 7: Analyze Results and Verify DeterministicModel Usage

In [ ]:
# Let's examine the trial data
print("Trial Results Summary:")
print("=" * 50)

best_trial = None
best_objective = float('inf')

for trial in experiment.trials.values():
    if trial.status.is_completed:
        params = trial.arm.parameters
        # In a real scenario, you'd fetch this from trial data
        results = evaluate_trial(params)
        
        print(f"Trial {trial.index}:")
        print(f"  Parameters: x={params['x']:.3f}, y={params['y']:.3f}")
        print(f"  Objective (analytical): {results['analytical_cost']:.3f}")
        print(f"  Constraint 1: {results['constraint_1']:.3f} (≤ 0.0)")
        print(f"  Constraint 2: {results['constraint_2']:.3f} (≤ 0.0)")
        
        # Check feasibility
        feasible = (results['constraint_1'] <= 0.0) and (results['constraint_2'] <= 0.0)
        print(f"  Feasible: {feasible}")
        
        if feasible and results['analytical_cost'] < best_objective:
            best_trial = trial
            best_objective = results['analytical_cost']
        
        print()

if best_trial:
    print(f"Best feasible trial: {best_trial.index}")
    print(f"Best objective value: {best_objective:.3f}")
    print(f"Best parameters: {best_trial.arm.parameters}")
else:
    print("No feasible trials found yet.")

print("\nNote: The analytical objective is computed deterministically,")
print("while constraints are modeled with Gaussian Processes.")

## Summary

This tutorial demonstrated how to use `DeterministicModel` in Ax for optimization problems with mixed analytical and black-box functions. Here are the key takeaways:

### What We Accomplished

1. **Defined an analytical objective function** (x² + y²) that we want to minimize
2. **Created black-box constraint functions** that simulate expensive evaluations
3. **Used `GenericDeterministicModel`** to wrap the analytical function
4. **Combined deterministic and probabilistic models** in a `ModelList`
5. **Integrated everything into Ax's workflow** using the modular BoTorch interface

### Key Benefits

- **Efficiency**: No need to learn a surrogate model for functions you already know analytically
- **Accuracy**: Analytical functions are evaluated exactly, not approximated
- **Flexibility**: Can mix deterministic and probabilistic models for different objectives/constraints
- **Integration**: Works seamlessly with Ax's optimization infrastructure

### When to Use This Approach

This approach is particularly useful when:
- You have analytical cost functions (monetary, computational, energy)
- Some objectives/constraints are cheap to evaluate analytically
- Others require expensive simulations or experiments
- You want to leverage known structure in your optimization problem

### Next Steps

To extend this example, you could:
- Add more complex analytical functions
- Use different acquisition functions optimized for mixed models
- Implement multi-objective optimization with mixed model types
- Add more sophisticated constraint handling

This minimal working example provides a foundation for incorporating analytical knowledge into your Bayesian optimization workflows with Ax and BoTorch.